# Counterfactual reasoning

Understanding what happened is valuable, but understanding what *could have* happened under different conditions is often more insightful. Counterfactual reasoning - the systematic exploration of "what if" scenarios - is how we learn which factors in a situation were causally decisive and which were incidental. By holding the actual outcome in mind while imagining plausible alternatives, we can ask: if this one condition had been different, would the outcome have changed? If the answer is yes, that condition was causally important. If no, the outcome was robust to that factor.

This mode of reasoning is central to how scientists evaluate hypotheses, how engineers run post-mortems, and how strategists stress-test plans. Applied to AI agents, it gives a model the ability to analyse a situation not just descriptively ("here is what happened") but causally ("here is *why* it happened and how fragile or robust the result was"). The pattern generates several plausible alternative conditions, reasons through each one independently to predict how the outcome would change, and then synthesises those comparisons into causal insights about the original situation.

In this notebook we implement counterfactual reasoning step by step - one function at a time - building toward a complete pipeline that identifies key conditions, imagines plausible alternatives, reasons through each, and arrives at a causal synthesis of the actual outcome.

In [1]:
import os
from collections import namedtuple
from typing import List

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

### Initialize the language model

In [2]:
# temperature=0.7 encourages varied counterfactual alternatives;
# scoring calls later use the same model instance
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.7)

## Representing a counterfactual
Each counterfactual needs to carry exactly three pieces of information: what condition was changed, what the outcome would likely have been under that change, and the causal explanation for why that outcome follows from the change. Three fields are sufficient - the original implementation also stored a `likelihood` label ("Low/Medium/High"), but that label was produced by the model with no rubric and added little analytical value. Replacing it with `causal_reasoning` gives us the *mechanism* rather than just a vague confidence tag, which is exactly what the synthesis step needs.

In [3]:
# Three fields — all populated at construction time, never modified afterwards
Counterfactual = namedtuple('Counterfactual', ['change', 'predicted_outcome', 'causal_reasoning'])
# change           — a concise description of the condition that was altered
#                    e.g. "the team had chosen a different launch date"
# predicted_outcome — one or two sentences on how the actual outcome would differ
# causal_reasoning  — the mechanism: *why* does this change lead to that outcome

Because `Counterfactual` is a `namedtuple`, a list of them can be iterated with `for cf in counterfactuals` and individual fields accessed as `cf.change`, `cf.predicted_outcome`, `cf.causal_reasoning` - all of which appear in the synthesis prompt and the display loop that follows. Immutability guarantees that nothing downstream can silently rewrite the reasoning after the fact.

## Generating counterfactual alternatives
The first step is identifying which conditions in the scenario were actually variable - which decisions could plausibly have gone differently, and which physical or contextual constraints were essentially fixed. We ask the model to read the full scenario (including the outcome) and propose plausible alternative conditions. Crucially, we pass the complete scenario rather than only the "situation" half: the model needs to know the actual outcome in order to vary the *right* conditions - the ones that plausibly affect it - rather than changing incidental details that would not have mattered.

In [4]:
def generate_counterfactuals(scenario: str, num_counterfactuals: int = 3) -> List[str]:
    """Return num_counterfactuals plain-text descriptions of alternative conditions."""
    # The full scenario (including outcome) is passed so the model can select
    # conditions that are actually relevant to what happened, not incidental details
    prompt = (
        f"Read the following scenario carefully. Identify {num_counterfactuals} key conditions "
        f"or decisions that shaped the outcome, and propose a plausible alternative for each.\n\n"
        f"Scenario: {scenario}\n\n"
        f"Format each alternative as a concise clause describing what could have been different, "
        f"without the 'What if' prefix. Start each clause with a subject such as 'the team', "
        f"'the company', 'the technology', 'the market', etc.\n"
        f"Example: '1. the team had chosen a waterfall process instead of iterative sprints'\n\n"
        f"Output only the numbered lines, nothing else."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    changes = []
    for line in raw.splitlines():
        line = line.strip()
        # Each valid line starts with a digit and contains a period separator
        if line and line[0].isdigit() and '.' in line:
            _, _, text = line.partition('.')   # drop the "N." prefix
            text = text.strip()
            if text:
                changes.append(text)

    # Fallback: if the model ignored formatting, use every non-empty line
    if not changes:
        changes = [l.strip() for l in raw.splitlines() if l.strip()]

    return changes[:num_counterfactuals]   # cap at the requested count

The prompt deliberately omits the "What if" prefix and asks for plain clauses. This avoids a roundtrip where the model generates `"What if the team had chosen X?"` and then the next function has to prepend `"What if"` again when constructing its own prompt. Each change is stored as a raw clause; the `"What if ..."` framing is added only where it appears in user-facing output. The line-by-line parser splits on the first period to discard the number prefix while preserving any periods inside the clause itself.

In [5]:
# Quick test: confirm the function returns well-formed alternative clauses
test_scenario = (
    "A software startup decided to pivot from B2B enterprise software to a B2C mobile app "
    "after 18 months of slow B2B sales. The pivot succeeded — they reached product-market "
    "fit within 6 months and grew to 1 million users."
)

test_changes = generate_counterfactuals(test_scenario, num_counterfactuals=3)
print(f"Scenario: {test_scenario}\n")
print("Counterfactual alternatives:")
for i, change in enumerate(test_changes, 1):
    print(f"  CF{i}: What if {change}?")

Scenario: A software startup decided to pivot from B2B enterprise software to a B2C mobile app after 18 months of slow B2B sales. The pivot succeeded — they reached product-market fit within 6 months and grew to 1 million users.

Counterfactual alternatives:
  CF1: What if the team had continued to refine their B2B product instead of pivoting to B2C.?
  CF2: What if the company had invested more in marketing and sales strategies for the B2B sector.?
  CF3: What if the market had shown stronger demand for B2B solutions, leading to increased sales opportunities.?


## Reasoning about a single alternative
Once we have the list of alternative conditions, we reason through each one independently. For a given change, we ask two questions: what would the outcome likely have been, and *why* - what is the causal mechanism that connects this change to the altered outcome? Asking for both in a single structured call is more efficient than making two separate LLM calls, and it forces the model to justify its predicted outcome rather than stating it as an assertion.

In [6]:
def reason_counterfactual(scenario: str, change: str) -> tuple:
    """Predict how a single alternative condition would alter the outcome.

    Returns (predicted_outcome: str, causal_reasoning: str).
    """
    # The full scenario is provided as context so the model knows both what
    # happened and what resulted — it cannot reason about alternatives without both
    prompt = (
        f"Consider the following scenario and a hypothetical alternative condition.\n\n"
        f"Actual scenario: {scenario}\n\n"
        f"Counterfactual: {change}?\n\n"
        f"Analyse what would likely have happened differently and explain the causal "
        f"mechanism — the chain of consequences that connects this change to the altered outcome.\n\n"
        f"Respond in exactly this format (two lines, nothing else):\n"
        f"OUTCOME: <one or two sentences describing the likely alternative outcome>\n"
        f"CAUSE: <one sentence explaining the causal mechanism behind this outcome>"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    raw = response.content.strip()

    predicted_outcome = raw   # fallback: use the full response if parsing fails
    causal_reasoning = ""     # empty string if CAUSE line is missing

    for line in raw.splitlines():
        line = line.strip()
        if line.upper().startswith('OUTCOME:'):
            predicted_outcome = line.split(':', 1)[1].strip()   # text after "OUTCOME:"
        elif line.upper().startswith('CAUSE:'):
            causal_reasoning = line.split(':', 1)[1].strip()    # text after "CAUSE:"

    return predicted_outcome, causal_reasoning

The two-field format - `OUTCOME` and `CAUSE` - separates prediction from explanation. `OUTCOME` answers "what would have happened?", while `CAUSE` answers "why would it have happened that way?" Both fields are stored in the `Counterfactual` record and later included verbatim in the synthesis prompt, so the causal mechanism is available to the final reasoning step rather than being discarded after display.

In [7]:
# Quick test: reason about one of the generated alternatives
test_outcome, test_cause = reason_counterfactual(test_scenario, test_changes[0])

print(f"Scenario: {test_scenario}\n")
print(f"Counterfactual: What if {test_changes[0]}?\n")
print(f"Predicted outcome: {test_outcome}")
print(f"Causal mechanism: {test_cause}")

Scenario: A software startup decided to pivot from B2B enterprise software to a B2C mobile app after 18 months of slow B2B sales. The pivot succeeded — they reached product-market fit within 6 months and grew to 1 million users.

Counterfactual: What if the team had continued to refine their B2B product instead of pivoting to B2C.?

Predicted outcome: The startup likely would have continued to struggle with slow sales and ultimately may have failed to gain traction in the B2B market, resulting in reduced funding and potential closure.
Causal mechanism: By not pivoting, the team would have remained focused on a product that lacked market demand, preventing them from discovering and addressing a more promising opportunity in the B2C space.


## Reasoning about all alternatives
With generation and individual reasoning in place, we wire them together. `test_counterfactuals` calls `reason_counterfactual` once per alternative and wraps each result in a `Counterfactual` namedtuple. Unlike hypothesis testing, there is no ranking step here: counterfactuals are not competing explanations that need to be sorted - they are independent probes of different dimensions of the scenario, and all of them contribute to the synthesis.

In [8]:
def test_counterfactuals(scenario: str, changes: List[str]) -> List[Counterfactual]:
    """Reason about each change independently and return Counterfactual namedtuples.

    Each call to reason_counterfactual sees only the scenario and one change —
    not the other alternatives — to prevent order-dependent anchoring.
    """
    counterfactuals = []
    for i, change in enumerate(changes, start=1):
        print(f"  Reasoning about alternative {i}/{len(changes)}...")   # progress indicator
        predicted_outcome, causal_reasoning = reason_counterfactual(scenario, change)
        # Assemble the immutable record; no modification after this point
        counterfactuals.append(Counterfactual(
            change=change,
            predicted_outcome=predicted_outcome,
            causal_reasoning=causal_reasoning,
        ))
    return counterfactuals   # returned in generation order — no sorting needed

Independent reasoning per alternative prevents the model from anchoring: each call to `reason_counterfactual` sees only the scenario and one change, not the other alternatives. This is the same design decision as in hypothesis testing - keeping evaluation independent avoids order-dependent bias where the model scores each alternative relative to the previous one rather than relative to the actual outcome.

## Synthesising causal insights
The synthesis step is what makes counterfactual reasoning more than just scenario generation. A list of "what if" predictions is descriptive; synthesis turns that list into a causal argument by asking: across all these alternatives, which changes produced the largest shifts in outcome, and why? That pattern of sensitivity - which conditions, when altered, reliably redirect the outcome - is the causal structure of the situation.

In [9]:
def synthesize(scenario: str, counterfactuals: List[Counterfactual]) -> str:
    """Compare the actual scenario with all alternatives to identify causal factors.

    The synthesis step is what distinguishes counterfactual reasoning from simple
    scenario generation: by examining which alternatives change the outcome and why,
    it identifies which conditions in the actual scenario were causally decisive.
    """
    # Build a compact summary that includes both the predicted outcome AND the causal
    # mechanism for each alternative — the mechanism is what the synthesis needs
    cf_summary = "\n".join(
        f"  {i+1}. If {cf.change}:\n"
        f"     Outcome: {cf.predicted_outcome}\n"
        f"     Because: {cf.causal_reasoning}"
        for i, cf in enumerate(counterfactuals)
    )
    prompt = (
        f"You have analysed a scenario and explored counterfactual alternatives.\n\n"
        f"Actual scenario: {scenario}\n\n"
        f"Counterfactual alternatives explored:\n{cf_summary}\n\n"
        f"Based on this comparison, write a synthesis that:\n"
        f"  - Identifies which conditions were causally decisive in producing the actual outcome\n"
        f"  - Assesses how robust or fragile the actual outcome was to changes\n"
        f"  - Draws at least one actionable lesson for future decisions in similar situations\n\n"
        f"Keep the synthesis to 3-5 sentences."
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()   # plain string; caller decides how to display it

The synthesis prompt receives both the `predicted_outcome` and `causal_reasoning` for each alternative. Including the causal mechanism - not just the predicted outcome - gives the synthesis model the material to reason about *why* some factors matter more than others, rather than just summarising what would have been different. This is the key design choice that separates causal analysis from descriptive comparison.

## The full counterfactual reasoning pipeline
With all four building blocks ready - `generate_counterfactuals`, `reason_counterfactual`, `test_counterfactuals`, and `synthesize` - the driver function simply chains them in sequence and prints progress at each step. It owns no state of its own; every intermediate value flows through as a function argument or return value.

In [10]:
def counterfactual_reason(scenario: str, num_counterfactuals: int = 3) -> dict:
    """Full pipeline: generate alternatives → reason about each → synthesise causal insights.

    Returns a dict with all intermediate data so callers can inspect any step.
    """
    print(f"Scenario: {scenario}\n")

    # Step 1 — Identify which conditions were variable and propose alternatives
    print(f"Step 1 — Generating {num_counterfactuals} counterfactual alternatives...")
    changes = generate_counterfactuals(scenario, num_counterfactuals)
    for i, change in enumerate(changes, 1):
        print(f"  CF{i}: What if {change}?")

    # Step 2 — For each alternative, predict the outcome and explain the mechanism
    print(f"\nStep 2 — Reasoning about each alternative...")
    counterfactuals = test_counterfactuals(scenario, changes)

    # Step 3 — Compare all alternatives to identify which factors were causally decisive
    print(f"\nStep 3 — Synthesising causal insights...")
    synthesis = synthesize(scenario, counterfactuals)

    return {
        "scenario":        scenario,
        "counterfactuals": counterfactuals,   # List[Counterfactual], in generation order
        "synthesis":       synthesis,
    }

Returning a plain dict means the results are immediately inspectable without importing any custom class. `result["counterfactuals"]` is a `List[Counterfactual]`, each element accessible by field name. `result["synthesis"]` is a plain string ready to display or log. The driver does not sort or filter the counterfactuals - they are returned in the order they were generated, which is also the order the synthesis prompt presents them to the model.

## Applying the full pipeline to a scenario
Let's run the complete pipeline on a scenario with a clear outcome, so we can observe how the synthesis identifies which factors were causally decisive. A traffic management deployment is a good test case: there are several plausible variables (technology choice, data quality, rollout scope, stakeholder buy-in) and the outcome metric is concrete, making it easier to judge whether the counterfactual reasoning is substantive.

In [11]:
scenario = (
    "A city government deployed a real-time traffic management system using AI-based signal "
    "timing. Within three months, average commute times fell by 18% and emergency vehicle "
    "response times improved by 25%. The system was adopted city-wide after the pilot."
)

result = counterfactual_reason(scenario, num_counterfactuals=3)

print("\n" + "=" * 65)
print("RESULTS")
print("=" * 65)

print("\nCounterfactual alternatives:\n")
for cf in result["counterfactuals"]:
    print(f"  What if {cf.change}?")
    print(f"  Outcome:  {cf.predicted_outcome}")
    print(f"  Because:  {cf.causal_reasoning}\n")

print("Causal synthesis:\n")
print(result["synthesis"])

Scenario: A city government deployed a real-time traffic management system using AI-based signal timing. Within three months, average commute times fell by 18% and emergency vehicle response times improved by 25%. The system was adopted city-wide after the pilot.

Step 1 — Generating 3 counterfactual alternatives...
  CF1: What if the city government had opted for a more gradual rollout instead of an immediate city-wide adoption.?
  CF2: What if the technology had included more robust data privacy measures from the outset.?
  CF3: What if the team had prioritized public feedback during the pilot phase to refine the system further.?

Step 2 — Reasoning about each alternative...
  Reasoning about alternative 1/3...
  Reasoning about alternative 2/3...
  Reasoning about alternative 3/3...

Step 3 — Synthesising causal insights...

RESULTS

Counterfactual alternatives:

  What if the city government had opted for a more gradual rollout instead of an immediate city-wide adoption.?
  Outcome

Passing the full `scenario` string (not just the situation half) to `counterfactual_reason` ensures the model has the actual outcome in view when selecting which alternatives to explore and when reasoning about each one. The plain-dict return value makes the result easy to inspect cell by cell: `result["counterfactuals"]` contains the ranked alternatives, `result["synthesis"]` contains the causal conclusion.

## Comparing counterfactual reasoning with direct analysis
Direct analysis answers the question "what happened and why?" in a single LLM call. It tends to produce a descriptive explanation that is consistent with the facts but rarely tests its own assumptions. Counterfactual reasoning answers the same question differently: it reveals which factors were *necessary* for the outcome by showing that removing them changes things, and which were merely *present* but not causally decisive. The comparison makes this gap concrete.

In [12]:
def direct_analyze(scenario: str) -> str:
    """Baseline: one LLM call, no counterfactual structure."""
    response = llm.invoke([HumanMessage(
        content=f"Analyze what happened in this scenario and explain why the outcome occurred: {scenario}"
    )])
    return response.content.strip()


comp_scenario = (
    "A product team launched a new feature without A/B testing. The feature was poorly received "
    "and user engagement dropped 20% in the first week. The team rolled back the feature after "
    "two weeks and engagement recovered."
)

print(f"Scenario: {comp_scenario}\n")

print("Direct analysis (1 LLM call):")
print(direct_analyze(comp_scenario))

print("\n" + "-" * 60)
print("Counterfactual reasoning (num_counterfactuals + 2 LLM calls):\n")
comp_result = counterfactual_reason(comp_scenario, num_counterfactuals=3)

print("\nCounterfactual alternatives explored:")
for cf in comp_result["counterfactuals"]:
    print(f"  — What if {cf.change}?")
    print(f"    → {cf.predicted_outcome}")

print("\nCausal synthesis:")
print(comp_result["synthesis"])

Scenario: A product team launched a new feature without A/B testing. The feature was poorly received and user engagement dropped 20% in the first week. The team rolled back the feature after two weeks and engagement recovered.

Direct analysis (1 LLM call):
In this scenario, the product team faced a significant challenge due to the launch of a new feature without conducting A/B testing. The outcome of poor reception and a subsequent drop in user engagement can be analyzed through several key factors:

1. **Lack of User Feedback**: A/B testing allows teams to gather data on how users respond to changes before a full rollout. By launching the feature without testing, the product team missed the opportunity to collect feedback from a subset of users. This feedback could have highlighted potential issues with the feature, allowing the team to make necessary adjustments before a broader release.

2. **Assumptions About User Needs**: The team likely made assumptions about user preferences or

**When to use counterfactual reasoning:** Post-mortem analysis where understanding *why* matters as much as understanding *what* - engineering incidents, product failures, strategy reviews. Risk assessment that requires knowing which assumptions the plan depends on. Decision evaluation where we want to judge a past choice not just on its outcome but on its robustness to uncertainty.

**Limitations to be aware of:** The quality of the analysis depends on the quality of the generated alternatives - if `generate_counterfactuals` produces superficial changes, the causal synthesis will be shallow. Each run costs `num_counterfactuals + 2` LLM calls (one to generate, one per alternative, one to synthesise). Hindsight bias is a structural risk: the model knows the actual outcome before generating alternatives, which may lead it to overweight factors that contributed to that outcome.

**Extending the pattern:** Ask the model to propose both "near-miss" alternatives (small changes) and "radical" alternatives (large changes) and compare whether the outcome is robust to small perturbations but not to large ones. Add a second round that takes the synthesis as the new scenario and asks what *that* outcome's key causal factors are - iterative counterfactual deepening.